In [1]:
import torch
import kornia.feature as KF
import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

HPATCHES_PATH = "../hpatches-sequences-release"

c:\Users\EasyMoneySniper\anaconda3\envs\nlp-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Check the layers of the LoFTR model
teacher = KF.LoFTR(pretrained='outdoor').eval()
for name, module in teacher.named_modules():
    print(name)


backbone
backbone.conv1
backbone.bn1
backbone.relu
backbone.layer1
backbone.layer1.0
backbone.layer1.0.conv1
backbone.layer1.0.conv2
backbone.layer1.0.bn1
backbone.layer1.0.bn2
backbone.layer1.0.relu
backbone.layer1.1
backbone.layer1.1.conv1
backbone.layer1.1.conv2
backbone.layer1.1.bn1
backbone.layer1.1.bn2
backbone.layer1.1.relu
backbone.layer2
backbone.layer2.0
backbone.layer2.0.conv1
backbone.layer2.0.conv2
backbone.layer2.0.bn1
backbone.layer2.0.bn2
backbone.layer2.0.relu
backbone.layer2.0.downsample
backbone.layer2.0.downsample.0
backbone.layer2.0.downsample.1
backbone.layer2.1
backbone.layer2.1.conv1
backbone.layer2.1.conv2
backbone.layer2.1.bn1
backbone.layer2.1.bn2
backbone.layer2.1.relu
backbone.layer3
backbone.layer3.0
backbone.layer3.0.conv1
backbone.layer3.0.conv2
backbone.layer3.0.bn1
backbone.layer3.0.bn2
backbone.layer3.0.relu
backbone.layer3.0.downsample
backbone.layer3.0.downsample.0
backbone.layer3.0.downsample.1
backbone.layer3.1
backbone.layer3.1.conv1
backbone.la

In [4]:
# loftr_coarse
# coarse_matching

In [ ]:
# Check the outputs of the coarse features and confidence matrix
teacher = KF.LoFTR(pretrained='outdoor').eval().cuda()

# use hooks to capture the outputs of the layers we care about
teacher_cache = {}
teacher.loftr_coarse.register_forward_hook(
    lambda m, inp, out: teacher_cache.update({'coarse_feat': out})
)
teacher.coarse_matching.register_forward_hook(
    lambda m, inp, out: teacher_cache.update({'conf_matrix': out})
)

# create dummy input images
img0 = torch.rand(1, 1, 480, 480).cuda()
img1 = torch.rand(1, 1, 480, 480).cuda()

with torch.no_grad():
    t_out = teacher({"image0": img0, "image1": img1})

# check what we captured in the cache
for key, val in teacher_cache.items():
    if isinstance(val, torch.Tensor):
        print(f"{key}: shape={val.shape}, dtype={val.dtype}")
    elif isinstance(val, (tuple, list)):
        for i, v in enumerate(val):
            if isinstance(v, torch.Tensor):
                print(f"{key}[{i}]: shape={v.shape}")
            else:
                print(f"{key}[{i}]: type={type(v)}")
    elif isinstance(val, dict):
        for k, v in val.items():
            if isinstance(v, torch.Tensor):
                print(f"{key}['{k}']: shape={v.shape}")
    else:
        print(f"{key}: type={type(val)}")

coarse_feat[0]: shape=torch.Size([1, 3600, 256])
coarse_feat[1]: shape=torch.Size([1, 3600, 256])
conf_matrix: type=<class 'NoneType'>


From the above result, we noticed that coarse_feat is valid, since it's a tuple with two tensors inside, which is just the coarse feature after img0 and img1 going through the transformer, shape is (B, 3600, 256) (matches: B, H*W, C), 3600 = 60 * 60, 256 is the shape of LoFTR coarse. 

However, conf_matrix has a shape of None. Since Kornia's coarse_matching does not return conf_matrix directly. We need to calculate conf_matrix from coarse_feat without hook. LofTR's dual-softmax was conducted based on the two featrues and then do softmax. 

In [ ]:
# We can see that the coarse features have shape (B, 3600, 256) and the confidence matrix has shape (B, 3600, 3600).
# The confidence matrix is computed from the coarse features, so we can verify that the teacher's confidence matrix is consistent with the coarse features.
# We can calculate the teacher confidence matrix from the coarse features and compare it with the one captured from the coarse_matching layer.
import torch.nn.functional as F
'''
This is the code to calculate the teacher confidence matrix from the coarse features. 
You can run this code to verify that the teacher's confidence matrix is consistent with the coarse features. 
The teacher confidence matrix should be similar to the one captured from the coarse_matching layer, which means our understanding of how the teacher computes the confidence matrix is correct.
'''
teacher.loftr_coarse.register_forward_hook(
    lambda m, inp, out: teacher_cache.update({
        'coarse_feat0': out[0],  # (B, 3600, 256)
        'coarse_feat1': out[1],  # (B, 3600, 256)
    })
)

# forward
with torch.no_grad():
    t_out = teacher({"image0": img0, "image1": img1})

# calculate the teacher confidence matrix
feat0 = F.normalize(teacher_cache['coarse_feat0'], dim=-1)
feat1 = F.normalize(teacher_cache['coarse_feat1'], dim=-1)
teacher_conf = torch.bmm(feat0, feat1.transpose(1, 2)) / 0.1  # temperature=0.1
teacher_conf = F.softmax(teacher_conf, dim=-1) * F.softmax(teacher_conf, dim=-2)
# teacher_conf: (B, 3600, 3600)

In [ ]:
'''
We need to make sure the student model's coarse features have the same shape as the teacher's coarse features, which is (B, 3600, 256).
Student: (B, 128, 60, 60) → (B, 3600, 128)
DO NOT RUN IT NOW!!! IT"S JUST A PSUEDO CODE TO SHOW THE IDEA. YOU NEED TO MODIFY THE STUDENT MODEL TO OUTPUT THE COARSE FEATURES IN THE SAME SHAPE AS THE TEACHER'S COARSE FEATURES.
'''

s_out = student({"image0": img0, "image1": img1})

s_feat = s_out['coarse_feat0'].flatten(2).permute(0, 2, 1)

# Projector: (B, 3600, 128) → (B, 3600, 256)
projector = nn.Linear(128, 256).cuda()
s_feat_proj = projector(s_feat)

# L_feature_KD
L_feat = F.mse_loss(s_feat_proj, teacher_cache['coarse_feat0'])

pass

In [ ]:
#Training Loop
## TODO: You need to implement the training loop to optimize the student model using the L_feature_KD loss. You can use any optimizer and learning rate scheduler you like. Make sure to backpropagate the loss and update the student model's parameters.